# eBOSS post-starburst selection

In this first notebook, we walk through the selection methods that allow us to take a parent sample of ~2 million galaxies from SDSS-IV/[eBOSS](https://www.sdss4.org/surveys/eboss/) and trim it down to a subsample of young and classic post-starburst candidates

In [2]:
import numpy as np               # to do math
from astropy.io import fits      # handline .fits files
import pandas as pd              # table format
import os                        # universal os
# CHANGES
# change #2

## Loading data

In [3]:
data_folder_name = 'data'  # folder containing the data file
filename = 'eboss_dr17_summary_odtrim8_v1.fit'  # data file

# accessing file from current directory
file_path = os.path.join('..', data_folder_name, filename)
file_path

'../data/eboss_dr17_summary_odtrim8_v1.fit'

To access the data, we will use `astropy` `fits` functions. Once opened, our data is stored in a "FITS table". The table header has info on the dimensions (# of rows and columns) and the column names. 

In [4]:
# hdu: "header data unit"
# contains information on the format and contents of our fits file
hdu = fits.open(file_path)

# displaying the header of our fits file
hdu[1].header

FileNotFoundError: [Errno 2] No such file or directory: '../data/eboss_dr17_summary_odtrim8_v1.fit'

As you can see, this table has a LOT of columns... to make things simple, we will extract the columns that we need and ignore the rest.

The main column values we are interested in are 

index measurements: 

- `DN4000` --> 4000 Angstrom break 
- `LICK_HDA` --> H-delta absorption
- `BALMERBREAK` --> Balmer break

their associated errors:

- `DN4000_ERR`
- `LICK_HDA_ERR`
- `BALMERBREAK_ERR`

line measurements:

- `H_ALPHA_EW`  --> strength of H-alpha emission
- `H_BETA_EW` --> strength of H-beta emission

and ionization source classification:

- `BPT_CLASS` --> classification on the BPT diagram

We will also save basic stuff about these eBOSS sources, such as redshift and identification information.

In [4]:
# ... this cell might take a minute to run ...

# initializing a DataFrame
eBOSS_catalog = pd.DataFrame()

# ------- setting the columns and rows for our table --------

# general info on objects
eBOSS_catalog['PLATE'] = list(hdu[1].data['PLATE'])                      
eBOSS_catalog['MJD'] = list(hdu[1].data['MJD'])  
eBOSS_catalog['FIBER'] = list(hdu[1].data['FIBER'])     
# PLATE, MJD, and FIBER numbers combined into an identification string
eBOSS_catalog['PMF_STRING'] = list(hdu[1].data['PMF_STRING'])    
# redshift of galaxy
eBOSS_catalog['Z'] = list(hdu[1].data['Z'])  
eBOSS_catalog['Z_ERR'] = list(hdu[1].data['Z_ERR'])  
# coordinates of galaxy 
eBOSS_catalog['RA'] = list(hdu[1].data['RA'])                                         
eBOSS_catalog['DEC'] = list(hdu[1].data['DEC']) 

# revelant measurements
eBOSS_catalog['H_ALPHA_EW'] = list(hdu[1].data['H_ALPHA_EW']) 
eBOSS_catalog['H_BETA_EW'] = list(hdu[1].data['H_BETA_EW']) 
eBOSS_catalog['OII_EW'] = list(hdu[1].data['OII_3727_EW'
eBOSS_catalog['D_4000'] = list(hdu[1].data['DN4000']) 
eBOSS_catalog['D_4000_ERR'] = list(hdu[1].data['DN4000_ERR'])
eBOSS_catalog['LICK_HDA'] = list(hdu[1].data['LICK_HDA'])
eBOSS_catalog['LICK_HDA_ERR'] = list(hdu[1].data['LICK_HDA_ERR'])
eBOSS_catalog['BALMERBREAK'] = list(hdu[1].data['BALMERBREAK'])
eBOSS_catalog['BALMERBREAK_ERR'] = list(hdu[1].data['BALMERBREAK_ERR'])
eBOSS_catalog['BPT_CLASS'] = list(hdu[1].data['BPT_CLASS'])

# --------------------------------------------------------------

# displaying our table
eBOSS_catalog

,PLATE,MJD,FIBER,PMF_STRING,Z,Z_ERR,RA,DEC,H_ALPHA_EW,H_BETA_EW,D_4000,D_4000_ERR,LICK_HDA,LICK_HDA_ERR,BALMERBREAK,BALMERBREAK_ERR,BPT_CLASS
0,3586,55181,2,03586-55181-0002,0.719691,0.000069,9.330078,-0.624116,-999.000000,6.830700,1.237226,0.042098,6.937127,1.299937,1.611019,0.080973,U
1,3586,55181,6,03586-55181-0006,0.533117,0.000141,9.409157,-0.250058,2.060369,0.239696,1.740146,0.079169,0.561692,1.792657,2.815180,0.180261,U
2,3586,55181,7,03586-55181-0007,0.474463,0.000134,9.360470,-0.212842,0.000000,0.510076,2.043482,0.137226,5.306952,2.099126,2.208125,0.129288,U
3,3586,55181,8,03586-55181-0008,0.489547,0.000159,9.351932,-0.159622,2.960478,0.000000,1.866135,0.130789,-2.729968,2.426148,2.312129,0.152515,U
4,3586,55181,9,03586-55181-0009,0.489842,0.000225,9.452027,-0.105271,1.556909,0.766571,1.835359,0.182474,-1.535746,3.165549,2.845022,0.305743,U
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1893690,12547,58928,985,12547-58928-0985,0.799789,0.000183,140.996347,2.055424,-999.000000,0.699077,1.875377,0.154568,2.356424,2.805292,2.392276,0.226185,U
1893691,12547,58928,986,12547-58928-0986,0.391036,0.000069,140.956507,2.087150,0.595381,0.788362,1.802910,0.067912,-1.160652,1.052939,1.982498,0.072412,U
1893692,12547,58928,988,12547-58928-0988,0.024241,0.000021,140.953350,2.107938,114.652946,26.197693,0.806632,0.180368,30.310261,7.886981,1.487158,0.719565,S
1893693,12547,58928,991,12547-58928-0991,0.535690,0.000092,140.905378,2.599161,34.857849,8.350409,1.340777,0.070753,2.616687,3.158643,1.942474,0.177461,L


Now we have a table with the data that we care about!

In [1]:
# Defining our 31 interesting galaxies
interesting_galaxies = [(8832, 918), (4221, 271), (11376, 234), (8497, 723), (7711, 557), (10750, 178), 
(9166, 760), (11114, 28), (10747, 766), (7388, 794), (7639, 684), (8233, 695), (8530, 222), 
(8491, 740), (8360, 94), (8381, 531), (10912, 812), (10232, 822), (10925, 294), (7572, 656), (11565, 847), (7683, 349),
(11336, 284), (7306, 754), (7735, 154), (7279, 396), (10938, 81), (6262, 911), (5967, 311), (8745, 354), (6453, 775)]

## Signal-to-noise and outlier cuts

We will set a reasonable signal-to-noise requirement for our sample ($\, S/N \geq 10$) and cut galaxies with unphysical values for any of our measurements.

In [5]:
final_quality_check = (
    (eBOSS_catalog['D_4000_ERR'] != 0) &
    (eBOSS_catalog['LICK_HDA_ERR'] != 0) &
    (eBOSS_catalog['BALMERBREAK_ERR'] > 0) &

    (eBOSS_catalog['D_4000'] / eBOSS_catalog['D_4000_ERR'] >= 10) &
    (eBOSS_catalog['BALMERBREAK'] / eBOSS_catalog['BALMERBREAK_ERR'] >= 10) &

    # Christy suggested we use a cut of LICK_HDA_ERR < 0.8, but in order to include all of our interesting galaxies,
    # I have bumped that cut to LICK_HDA_ERR < 1.5. This can be changed if needed later on.
    (eBOSS_catalog['LICK_HDA_ERR'] < 1.5) &
    (
        ((eBOSS_catalog['Z'] <= 0.49) & (eBOSS_catalog['H_ALPHA_EW'] != -999)) | 
        ((eBOSS_catalog['Z'] > 0.49) & ((eBOSS_catalog['H_BETA_EW'] != -999) | (eBOSS_catalog['OII_EW'] != -999)))
    )
)

eBOSS_catalog = eBOSS_catalog[final_quality_check]

print(f"Total rows remaining in final sample: {len(eBOSS_catalog)}")
final_matches = eBOSS_catalog[eBOSS_catalog.set_index(['PLATE', 'FIBER']).index.isin(interesting_galaxies)]
print(f"Total interesting galaxies found in final sample after signal-to-noise and invalid data cuts: {len(final_matches)} out of 31 interesting galaxies.")

5141

Let's also remove any outliers in our dataset

In [6]:
# identifying outliers 
outlier_mask = (

    # masking out ouliers
    (eBOSS_catalog['H_ALPHA_EW'] > 0) & (eBOSS_catalog['H_ALPHA_EW'] < 100) & 
    (eBOSS_catalog['H_BETA_EW'] > 0) & (eBOSS_catalog['H_BETA_EW'] < 100) & 
    (eBOSS_catalog['LICK_HDA'] > -5) & (eBOSS_catalog['LICK_HDA'] < 13) &
    (eBOSS_catalog['D_4000'] > 0) & (eBOSS_catalog['D_4000'] < 2.5) & 
    (eBOSS_catalog['BALMERBREAK'] > 0) & (eBOSS_catalog['BALMERBREAK'] < 3.5) 
)

eBOSS_catalog = eBOSS_catalog[outlier_mask]
len(eBOSS_catalog)  # let's see the size of this new sample

3919

## Selection process 

We will select post-starburst galaxies using relationships between spectral features. Below is a starting point. We experiment with different selection methods as well.

The selection criteria for classic post-starburst galaxies are taken from Chen+2019 (see the figure below - the region outlined in black represents this cut). And the selection criterial for young post-starbursts are taken from Prof. Christy Tremonti's experimentations.

<img src="../plots/Chen+2019_figure.png" width="600">

In [ ]:
# taken from Chen+2019 (region outlined in black)
classic_psb_mask = (

    (eBOSS_catalog['LICK_HDA'] > 3.0) &
    (eBOSS_catalog['H_ALPHA_EW'] < 10.0) &
    (np.log10(eBOSS_catalog['H_ALPHA_EW']) < (0.23 * eBOSS_catalog['LICK_HDA'] - 0.46))
)

# Christy provided these criteria (experimentation)
young_psb_mask = (
    
    (eBOSS_catalog['BALMERBREAK'] > 0.9) & (eBOSS_catalog['BALMERBREAK'] < 1.9) &
    (eBOSS_catalog['LICK_HDA'] > (eBOSS_catalog['BALMERBREAK'] * 5.2 - 4.7)) & 
    (eBOSS_catalog['BALMERBREAK'] > (eBOSS_catalog['D_4000'] * 7.0 - 6.5)) & 
    (eBOSS_catalog['H_BETA_EW'] < 7) # same effect as H-alpha
)

# adding a classification column to our dataframe and applying our masks
eBOSS_catalog['PSB_CLASS'] = 'starburst'
eBOSS_catalog.loc[classic_psb_mask, 'PSB_CLASS'] = 'classic PSB'
eBOSS_catalog.loc[young_psb_mask, 'PSB_CLASS'] = 'young PSB'     # note that any galaxies that satisfy both classic and young will be classified as young

# saving our final dataframe to a .csv file 
eBOSS_catalog.to_csv('../classified_sample/classified_eBOSS_sample.csv', index=False)

We have saved our sample measurements and classifications in a .csv file and will use it in the next notebook: `02_eBOSS_data_analysis.ipynb`.

To experiement with different classification criteria, you can make changes to the cell above and a new .csv file will replace the current one.